# Text2SQL Benchmark

Đo hiệu năng sinh SQL từ câu hỏi tiếng Việt:
- `llm_time` / `e2e_time`: thời gian invoke và end-to-end (kèm chạy DB)
- `ttft`: time-to-first-token khi streaming

Schema dùng bản **compact** (khớp với DB thật `agent_pm`) để giảm token và TTFT.

In [ ]:
import re
import time
import statistics
from typing import List, Dict, Any

from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_core.messages import SystemMessage, HumanMessage

# ========================= CONFIG =========================
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "agent_pm"

DB_URI = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

API_KEY=""
MODEL="claude-haiku-4-5-20251001"
BASE_URL="https://api.freemodel.dev/v1"


TEMPERATURE = 0
MAX_TOKENS = 512
COMPANY_ID = 1
USER_ID = 1
TOP_K = 5

# ========================= DB =========================
db = SQLDatabase.from_uri(DB_URI, sample_rows_in_table_info=3)
print(f"Connected to {DB_NAME} ({db.dialect})")

Connected to agent_pm (postgresql)


In [5]:
# Compact schema khớp với DB thật (đã introspect). 'backlogs' = timesheet/worklog.
SCHEMA_COMPACT = """
users(id, full_name, email, role, department, position, active, company_id)
companies(id, name, code, currency_id)
projects(id, name, code, status, priority, start_date, end_date, total_hours,
         task_count, member_count, owner_id, currency_id, company_id)
members(id, project_id, user_id, role, joined_at)
tasks(id, name, status, priority, deadline, end_at, total_hours,
      project_id, assignee_id, milestone_id, company_id)
milestones(id, name, status, due_date, completion_pct, task_count, done_count, project_id)
backlogs(id, status, source, work_date, hours, task_id, project_id, user_id,
         currency_id, approver_id, approved_at, company_id)
task_blockers(id, task_id, severity, description, resolved_at, created_at)
scopes(id, name, estimated_hours, project_id, task_id, assignee_id)
currencies(id, code, symbol, rate)
"""

SYSTEM_PROMPT = f"""
Bạn là Text2SQL Agent cho database PostgreSQL `agent_pm`.

QUY TẮC OUTPUT:
- Chỉ trả về DUY NHẤT 1 câu SQL, không markdown, không giải thích.
- Chỉ dùng SELECT hoặc WITH ... SELECT. Cấm INSERT/UPDATE/DELETE/DROP/ALTER/TRUNCATE/CREATE.
- SQL kết thúc bằng dấu chấm phẩy ';'.
- Không SELECT *. Thêm LIMIT {TOP_K} cho query non-aggregate (trừ khi đã có LIMIT).
- Non-aggregate phải có ORDER BY cột ổn định.

MULTI-TENANCY:
- Mọi query đụng users/projects/tasks/backlogs phải filter company_id = {COMPANY_ID}.
- Bảng phụ (members/milestones/task_blockers/scopes) kế thừa qua JOIN projects/tasks.
- 'của tôi' => user_id = {USER_ID}.

ENUM (bắt buộc cast):
- TaskStatus: TODO, IN_PROGRESS, REVIEW, DONE        -> t.status = 'DONE'::\"TaskStatus\"
- ProjectStatus: PLANNED, IN_PROGRESS, ON_HOLD, COMPLETED, CANCELLED
- Priority: LOW, MEDIUM, HIGH, URGENT                -> p.priority = 'HIGH'::\"Priority\"
- BacklogStatus: PENDING, APPROVED, REJECTED
- BlockerSeverity: LOW, MED, HIGH
- Role: ADMIN, MANAGER, MEMBER, VIEWER

THỜI GIAN (Asia/Ho_Chi_Minh, không dùng CURRENT_DATE/NOW() thuần):
- Hôm nay: (NOW() AT TIME ZONE 'Asia/Ho_Chi_Minh')::date
- N ngày tới: deadline BETWEEN today AND today + INTERVAL 'N days'

Nếu câu hỏi ngoài phạm vi DB: SELECT 'INVALID_QUESTION' AS error;

Schema:
{SCHEMA_COMPACT}
"""

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

System prompt: 2101 chars


In [6]:
llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

stream_llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    streaming=True,
)

# Kiểm tra model thật mà proxy phục vụ (freemodel.dev có thể route sang model khác)
_probe = llm.invoke("ping")
print("served model:", _probe.response_metadata.get("model_name"))

served model: gpt-5.4


In [7]:
def clean_sql(text: str) -> str:
    text = text.strip()
    text = re.sub(r"```sql", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text).strip()
    if not text.endswith(";"):
        text += ";"
    return text


def validate_sql(sql: str) -> bool:
    s = sql.lower().strip()
    forbidden = ["insert ", "update ", "delete ", "drop ", "alter ", "truncate ", "create "]
    if any(w in s for w in forbidden):
        return False
    return s.startswith("select") or s.startswith("with")


def print_section(title: str):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

In [8]:
async def generate_sql(question: str) -> Dict[str, Any]:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question),
    ]

    t0 = time.perf_counter()
    response = await llm.ainvoke(messages)
    t1 = time.perf_counter()

    sql = clean_sql(response.content)
    t2 = time.perf_counter()

    return {
        "question": question,
        "sql": sql,
        "valid": validate_sql(sql),
        "llm_time": t1 - t0,
        "postprocess_time": t2 - t1,
        "total_generate_time": t2 - t0,
        "raw_response": response.content,
        "usage": getattr(response, "usage_metadata", None),
    }


async def run_text2sql(question: str, execute_db: bool = True) -> Dict[str, Any]:
    total_start = time.perf_counter()
    gen = await generate_sql(question)

    db_time = db_result = db_error = None
    if execute_db and gen["valid"]:
        db_start = time.perf_counter()
        try:
            db_result = db.run(gen["sql"])
        except Exception as e:
            db_error = str(e)
        db_time = time.perf_counter() - db_start

    return {
        **gen,
        "db_time": db_time,
        "db_result": db_result,
        "db_error": db_error,
        "e2e_time": time.perf_counter() - total_start,
    }

In [9]:
async def stream_generate_sql(question: str) -> Dict[str, Any]:
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=question),
    ]

    full_text = ""
    start = time.perf_counter()
    first_token_time = None

    async for chunk in stream_llm.astream(messages):
        if first_token_time is None:
            first_token_time = time.perf_counter()
        if chunk.content:
            full_text += chunk.content
            print(chunk.content, end="", flush=True)

    end = time.perf_counter()
    sql = clean_sql(full_text)

    return {
        "question": question,
        "sql": sql,
        "valid": validate_sql(sql),
        "ttft": (first_token_time - start) if first_token_time else None,
        "stream_total_time": end - start,
        "raw_response": full_text,
    }

In [10]:
async def benchmark_question(question: str, runs: int = 5, execute_db: bool = True) -> Dict[str, Any]:
    results = []
    print_section(f"BENCHMARK: {question}")

    for i in range(runs):
        print(f"\nRun {i + 1}/{runs}")
        r = await run_text2sql(question, execute_db=execute_db)
        results.append(r)
        print("SQL:", r["sql"])
        print(f"LLM time: {r['llm_time']:.3f}s", end="")
        if r["db_time"] is not None:
            print(f" | DB time: {r['db_time']:.3f}s", end="")
        print(f" | E2E: {r['e2e_time']:.3f}s")
        if r["db_error"]:
            print("DB error:", r["db_error"])

    llm_times = [r["llm_time"] for r in results]
    e2e_times = [r["e2e_time"] for r in results]
    db_times = [r["db_time"] for r in results if r["db_time"] is not None]

    summary = {
        "question": question,
        "runs": runs,
        "llm_avg": statistics.mean(llm_times),
        "llm_min": min(llm_times),
        "llm_max": max(llm_times),
        "e2e_avg": statistics.mean(e2e_times),
        "db_avg": statistics.mean(db_times) if db_times else None,
        "results": results,
    }

    print_section("SUMMARY")
    print(f"LLM avg/min/max: {summary['llm_avg']:.3f} / {summary['llm_min']:.3f} / {summary['llm_max']:.3f}s")
    print(f"E2E avg: {summary['e2e_avg']:.3f}s")
    if summary["db_avg"] is not None:
        print(f"DB avg: {summary['db_avg']:.3f}s")
    return summary


async def benchmark_many_questions(questions: List[str], runs: int = 3, execute_db: bool = True):
    all_results = []
    for q in questions:
        all_results.append(await benchmark_question(q, runs=runs, execute_db=execute_db))

    print_section("FINAL SUMMARY")
    for item in all_results:
        print(f"\n[{item['llm_avg']:.3f}s LLM | {item['e2e_avg']:.3f}s E2E] {item['question']}")
    return all_results

In [11]:
questions = [
    "Hiện tại tôi có bao nhiêu dự án?",
    "Danh sách dự án đang chạy là gì?",
    "Task nào đến hạn trong 7 ngày tới?",
    "Worklog của tôi hôm nay là gì?",
    "Dự án nào có độ ưu tiên cao?",
]

## 1. Test 1 câu (invoke + chạy DB)

In [12]:
result = await run_text2sql("Hiện tại tôi có bao nhiêu dự án?", execute_db=True)

print_section("SINGLE RESULT")
print("SQL:", result["sql"])
print("DB result:", result["db_result"])
print("DB error:", result["db_error"])
print(f"LLM time: {result['llm_time']:.3f}s")
print(f"DB time: {result['db_time']:.3f}s" if result["db_time"] else "DB time: -")
print(f"E2E time: {result['e2e_time']:.3f}s")


SINGLE RESULT
SQL: SELECT COUNT(*) AS project_count FROM projects p WHERE p.company_id = 1;
DB result: [(8,)]
DB error: None
LLM time: 7.296s
DB time: 0.003s
E2E time: 7.300s


## 2. Benchmark nhiều lần (1 câu)

In [13]:
summary = await benchmark_question("Hiện tại tôi có bao nhiêu dự án?", runs=5, execute_db=True)


BENCHMARK: Hiện tại tôi có bao nhiêu dự án?

Run 1/5
SQL: SELECT COUNT(p.id) AS project_count FROM projects p WHERE p.company_id = 1 AND p.owner_id = 1;
LLM time: 70.686s | DB time: 0.003s | E2E: 70.690s

Run 2/5
SQL: SELECT COUNT(*) AS project_count FROM projects p WHERE p.company_id = 1 AND (p.owner_id = 1 OR EXISTS (SELECT 1 FROM members m WHERE m.project_id = p.id AND m.user_id = 1));
LLM time: 46.179s | DB time: 0.003s | E2E: 46.183s

Run 3/5
SQL: SELECT COUNT(p.id) AS project_count FROM projects p WHERE p.company_id = 1;
LLM time: 5.843s | DB time: 0.002s | E2E: 5.846s

Run 4/5
SQL: SELECT COUNT(p.id) AS project_count FROM projects p WHERE p.company_id = 1;
LLM time: 7.984s | DB time: 0.005s | E2E: 7.989s

Run 5/5
SQL: SELECT COUNT(*) AS project_count FROM projects p WHERE p.company_id = 1;
LLM time: 4.299s | DB time: 0.003s | E2E: 4.302s

SUMMARY
LLM avg/min/max: 26.998 / 4.299 / 70.686s
E2E avg: 27.002s
DB avg: 0.003s


## 3. Streaming / TTFT

In [14]:
print_section("STREAMING TEST")
stream_result = await stream_generate_sql("Danh sách task quá hạn của tôi là gì?")

print("\n")
print(f"TTFT: {stream_result['ttft']:.3f}s")
print(f"Stream total: {stream_result['stream_total_time']:.3f}s")
print("SQL:", stream_result["sql"])


STREAMING TEST
SELECT t.id, t.name, t.status, t.priority, t.deadline, t.project_id FROM tasks t WHERE t.company_id = 1 AND t.assignee_id = 1 AND t.deadline IS NOT NULL AND t.deadline < (NOW() AT TIME ZONE 'Asia/Ho_Chi_Minh')::date AND t.status <> 'DONE'::"TaskStatus" ORDER BY t.deadline ASC, t.id ASC LIMIT 5;

TTFT: 3.846s
Stream total: 13.488s
SQL: SELECT t.id, t.name, t.status, t.priority, t.deadline, t.project_id FROM tasks t WHERE t.company_id = 1 AND t.assignee_id = 1 AND t.deadline IS NOT NULL AND t.deadline < (NOW() AT TIME ZONE 'Asia/Ho_Chi_Minh')::date AND t.status <> 'DONE'::"TaskStatus" ORDER BY t.deadline ASC, t.id ASC LIMIT 5;


## 4. Benchmark nhiều câu

In [15]:
all_results = await benchmark_many_questions(questions, runs=3, execute_db=True)


BENCHMARK: Hiện tại tôi có bao nhiêu dự án?

Run 1/3
SQL: SELECT COUNT(*) AS project_count FROM projects p WHERE p.company_id = 1 AND p.owner_id = 1;
LLM time: 13.609s | DB time: 0.004s | E2E: 13.613s

Run 2/3
SQL: SELECT COUNT(DISTINCT p.id) AS project_count FROM projects p LEFT JOIN members m ON m.project_id = p.id AND m.user_id = 1 WHERE p.company_id = 1 AND (p.owner_id = 1 OR m.user_id = 1);
LLM time: 31.208s | DB time: 0.005s | E2E: 31.213s

Run 3/3
SQL: SELECT COUNT(*) AS project_count FROM projects p WHERE p.company_id = 1 AND p.owner_id = 1 AND p.status IN ('PLANNED'::"ProjectStatus", 'IN_PROGRESS'::"ProjectStatus", 'ON_HOLD'::"ProjectStatus");
LLM time: 10.909s | DB time: 0.004s | E2E: 10.913s
DB error: (psycopg2.errors.InvalidTextRepresentation) invalid input value for enum "ProjectStatus": "ON_HOLD"
LINE 1: ...:"ProjectStatus", 'IN_PROGRESS'::"ProjectStatus", 'ON_HOLD':...
                                                             ^

[SQL: SELECT COUNT(*) AS project_count